<a href="https://colab.research.google.com/github/SrijaMadarapu8/AI-ML/blob/main/fastapi_backend.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#FastAPI Backend Wrapping Claude + OpenAI

A backend service with one endpoint that accepts a prompt and a
`model_provider` choice, calls the corresponding frontier model API, and
returns a structured response with latency, retry, and fallback handling.

In [1]:
!pip install fastapi uvicorn anthropic openai google-generativeai pydantic nest_asyncio tenacity requests -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 13.1 MB/s eta 0:00:00


In [2]:
from google.colab import userdata
import os
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
os.environ["Gemini_API_Key"] = userdata.get("Gemini_API_Key")

## Define request/response models and the FastAPI app

In [3]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import time, logging
from tenacity import retry, stop_after_attempt, wait_fixed, retry_if_exception_type
from anthropic import Anthropic
from openai import OpenAI as OpenAIClient
import google.generativeai as genai

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("llm-backend")

anthropic_client = Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
openai_client = OpenAIClient(base_url="https://agentate-model.services.ai.azure.com/openai/v1", api_key=os.environ["OPENAI_API_KEY"])
genai.configure(api_key=os.environ["Gemini_API_Key"])
gemini_model = genai.GenerativeModel("gemini-2.0-flash")

app = FastAPI(title="LLM Backend Demo")

class QueryRequest(BaseModel):
    prompt: str
    model_provider: str  # "claude", "openai", or "gemini"

class QueryResponse(BaseModel):
    answer: str
    provider_used: str
    latency_ms: float

@retry(stop=stop_after_attempt(3), wait=wait_fixed(1),
       retry=retry_if_exception_type(Exception), reraise=True)
def call_claude(prompt: str) -> str:
    resp = anthropic_client.messages.create(
        model="claude-sonnet-4-5",
        max_tokens=300,
        messages=[{"role": "user", "content": prompt}],
    )
    return resp.content[0].text

@retry(stop=stop_after_attempt(3), wait=wait_fixed(1),
       retry=retry_if_exception_type(Exception), reraise=True)
def call_openai(prompt: str) -> str:
    resp = openai_client.chat.completions.create(
        model="gpt-5.4-nano",
        messages=[{"role": "user", "content": prompt}],
    )
    return resp.choices[0].message.content

@retry(stop=stop_after_attempt(3), wait=wait_fixed(1),
       retry=retry_if_exception_type(Exception), reraise=True)
def call_gemini(prompt: str) -> str:
    resp = gemini_model.generate_content(prompt)
    return resp.text

PROVIDERS = {
    "claude": call_claude,
    "openai": call_openai,
    "gemini": call_gemini,
}

@app.post("/query", response_model=QueryResponse)
def query(req: QueryRequest):
    start = time.time()
    provider_used = req.model_provider

    if req.model_provider not in PROVIDERS:
        raise HTTPException(status_code=400, detail=f"model_provider must be one of {list(PROVIDERS.keys())}")

    try:
        answer = PROVIDERS[req.model_provider](req.prompt)
    except Exception as primary_err:
        logger.warning(f"Primary provider {req.model_provider} failed: {primary_err}. Falling back to gemini.")
        fallback = "gemini" if req.model_provider != "gemini" else "groq_unavailable"
        try:
            answer = PROVIDERS[fallback](req.prompt)
            provider_used = fallback
        except Exception as fallback_err:
            logger.error(f"Fallback provider {fallback} also failed: {fallback_err}")
            raise HTTPException(status_code=502, detail="Both primary and fallback providers failed")

    latency_ms = (time.time() - start) * 1000
    logger.info(f"provider={provider_used} latency_ms={latency_ms:.1f}")
    return QueryResponse(answer=answer, provider_used=provider_used, latency_ms=latency_ms)

@app.get("/health")
def health():
    return {"status": "ok"}

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


## Run the server in the background inside Colab
Colab is a single notebook process, so `uvicorn.run()` needs to run on a
background thread rather than blocking the cell forever.

In [4]:
import threading
import uvicorn
import nest_asyncio

nest_asyncio.apply()

config = uvicorn.Config(app, host="127.0.0.1", port=8000, log_level="warning")
server = uvicorn.Server(config)

def run_server():
    server.run()

thread = threading.Thread(target=run_server, daemon=True)
thread.start()

import time
time.sleep(2)  # give the server a moment to start
print("Server should now be running at http://127.0.0.1:8000")

Server should now be running at http://127.0.0.1:8000


## Test the endpoint

In [6]:
import requests

resp = requests.get("http://127.0.0.1:8000/health")
print(resp.json())

{'status': 'ok'}


In [7]:
resp = requests.post(
    "http://127.0.0.1:8000/query",
    json={"prompt": "In one sentence, what is retrieval-augmented generation?", "model_provider": "claude"}
)
print(resp.json())

ERROR:llm-backend:Fallback provider gemini also failed: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash
Please retry in 6.642349914s.


{'detail': 'Both primary and fallback providers failed'}


In [8]:
resp = requests.post(
    "http://127.0.0.1:8000/query",
    json={"prompt": "In one sentence, what is retrieval-augmented generation?", "model_provider": "openai"}
)
print(resp.json())

{'answer': 'Retrieval-augmented generation (RAG) is a method where a model first retrieves relevant information from external sources and then uses that retrieved context to generate more accurate, grounded responses.', 'provider_used': 'openai', 'latency_ms': 2252.3138523101807}


In [ ]:
resp = requests.post(
    "http://127.0.0.1:8000/query",
    json={"prompt": "In one sentence, what is retrieval-augmented generation?", "model_provider": "gemini"}
)
print(resp.json())

ERROR:llm-backend:Fallback provider claude also failed: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'Your credit balance is too low to access the Anthropic API. Please go to Plans & Billing to upgrade or purchase credits.'}, 'request_id': 'req_011CdHh3RZKacZ1auYyvBQDw'}


{'detail': 'Both providers failed'}


## Test the fallback path
Pass an invalid provider name to see the structured 400 error, and inspect
the logs above to see the retry/fallback behavior when a provider call
fails.

In [ ]:
resp = requests.post(
    "http://127.0.0.1:8000/query",
    json={"prompt": "hello", "model_provider": "not_a_real_provider"}
)
print(resp.status_code, resp.json())